In [1]:
import numpy as np
from astropy.io import fits
import matplotlib
from scipy.ndimage import rotate
matplotlib.use('TkAgg')
import matplotlib.pyplot as plt
from matplotlib.widgets import Slider
from matplotlib.colors import LogNorm

def load_data():
    left  = fits.getdata('HD__36112_2020-12-19_cube_left_frames.fits')
    right = fits.getdata('HD__36112_2020-12-19_cube_right_frames.fits')
    p = fits.getdata('HD__36112_2020-12-19_parangs.fits')
    return left.astype(np.float64), right.astype(np.float64), p

LEFT, RIGHT, p = load_data()
_, NY, NX = LEFT.shape
N_CYCLES = LEFT.shape[0] // 4
Y0, X0 = (NY - 1) / 2, (NX - 1) / 2

EPS = 1e-12

indices = np.array([1, 2, 3, 4] * 35)
cube_single_sum = LEFT + RIGHT
cube_single_difference = LEFT - RIGHT


cube_I_Q_double_sum = 0.5*(cube_single_sum[indices == 1, :, :] + cube_single_sum[indices == 2, :, :])
cube_I_U_double_sum = 0.5*(cube_single_sum[indices == 3, :, :] + cube_single_sum[indices == 4, :, :])

    

In [3]:
cube_Q_double_difference = 0.5*(cube_single_difference[indices == 1, :, :] - cube_single_difference[indices == 2, :, :])
cube_U_double_difference = 0.5*(cube_single_difference[indices == 3, :, :] - cube_single_difference[indices == 4, :, :])

In [4]:
# Define pupil-offset (deg) in pupil-tracking mode (SPHERE User Manual P99.0, 6th public release, P99 Phase 1)
pupil_offset = 135.99

# Define true North correction (deg) (SPHERE User Manual P99.0, 6th public release, P99 Phase 1)
true_north_correction = -1.75

def compute_mean_angle(angles, degree_radian='degree', axis=None):
    '''
    Calculate mean of angles using mean of circular quantities

    Input:
        angles: list or array of angles
        degree_radian: if 'degree' input and output are in degree; else in radians
        axis: axis to compute mean over; None or int or tuple of ints (default = None)

    Output:
        mean_angle: scalar of mean angle

    File written by Rob van Holstein
    Function status: verified
    '''

    # Convert angle to rad if specified in degree
    if degree_radian == 'degree':
        angles = np.deg2rad(angles)

    # Compute mean angle
    y = np.mean(np.sin(angles), axis = axis)
    x = np.mean(np.cos(angles), axis = axis)
    mean_angle = np.arctan2(y, x)

    # Convert mean angle to degree if input angle specified in degree
    if degree_radian == 'degree':
        mean_angle = np.rad2deg(mean_angle)

    return mean_angle

In [5]:
p_Q = compute_mean_angle(np.vstack([p[indices == 1], p[indices == 2]]), axis = 0)
p_U = compute_mean_angle(np.vstack([p[indices == 3], p[indices == 4]]), axis = 0)

rotation_angles_Q = -p_Q - pupil_offset - true_north_correction
rotation_angles_U = -p_U - pupil_offset - true_north_correction

In [6]:
cube_Q_IP_subtracted = cube_Q_double_difference #- IP_Q[:, np.newaxis, np.newaxis]*cube_I_Q_double_sum
cube_U_IP_subtracted = cube_U_double_difference #- IP_U[:, np.newaxis, np.newaxis]*cube_I_U_double_sum

# Derotate IP-subtracted Q- and U-images
cube_Q_derotated = np.zeros(cube_Q_IP_subtracted.shape)
cube_U_derotated = np.zeros(cube_U_IP_subtracted.shape)
    
for i, (frame_Q, rotation_angle_Q) in enumerate(zip(cube_Q_IP_subtracted, rotation_angles_Q)):
    cube_Q_derotated[i, :, :] = rotate(frame_Q, rotation_angle_Q, reshape=False)

for i, (frame_U, rotation_angle_U) in enumerate(zip(cube_U_IP_subtracted, rotation_angles_U)):
    cube_U_derotated[i, :, :] = rotate(frame_U, rotation_angle_U, reshape=False)

In [7]:
ALL_Q = cube_Q_derotated
ALL_U = cube_U_derotated

In [8]:
def compute_azimuthal_stokes_parameters(frame_Q, frame_U, rotation_angle=0, center_coordinates=None):
    '''
    Compute images of azimuthal stokes parameters Q_phi and U_phi using defintions of
    de Boer et al. (2020)

    Input:
        frame_Q: Stokes Q-image
        frame_U: Stokes U-image
        rotation_angle: additional image rotation (deg)
        center_coordinates: tuple containing center coordinates in x and y; if
            None, use center of frame

    Output:
        Q_phi: azimuthal Stokes Q_phi-image
        U_phi: azimuthal Stokes U_phi-image
        phi: frame showing values of the angle phi (deg)

    Note:
        Q_phi > 0 for azimuthal polarization and Q_phi < 0 for radial polarization

    File written by Rob van Holstein
    Function status: verified
    '''

    # Create grid and compute angle
    x = np.arange(0, frame_Q.shape[-1])
    y = np.arange(0, frame_Q.shape[-2])
    xm, ym = np.meshgrid(x, y)

    if center_coordinates is None:
        x_center = 0.5*x[-1]
        y_center = 0.5*y[-1]
    else:
        x_center = center_coordinates[0]
        y_center = center_coordinates[1]

    phi = np.arctan2((x_center - xm), (ym - y_center)) + np.deg2rad(rotation_angle)

    # Compute Q_phi- and U_phi-images
    frame_Q_phi = -frame_Q*np.cos(2*phi) - frame_U*np.sin(2*phi)
    frame_U_phi = frame_Q*np.sin(2*phi) - frame_U*np.cos(2*phi)

    # Convert frame showing phi to degrees
    phi = np.rad2deg(phi)

    return frame_Q_phi, frame_U_phi, phi

In [9]:
ALL_Qphi = np.array([compute_azimuthal_stokes_parameters(ALL_Q[i], ALL_U[i])[0] for i in range(len(ALL_Q))])
ALL_Uphi = np.array([compute_azimuthal_stokes_parameters(ALL_Q[i], ALL_U[i])[1] for i in range(len(ALL_Q))])

In [10]:
ALL_Qphi[2]

array([[ 0.,  0.,  0., ..., -0., -0.,  0.],
       [ 0.,  0.,  0., ..., -0.,  0.,  0.],
       [ 0.,  0.,  0., ...,  0.,  0.,  0.],
       ...,
       [ 0.,  0., -0., ...,  0.,  0.,  0.],
       [ 0., -0., -0., ...,  0.,  0.,  0.],
       [-0., -0., -0., ...,  0.,  0.,  0.]], shape=(124, 124))

In [14]:
fig, ax = plt.subplots(figsize=(7, 6))
plt.subplots_adjust(bottom=0.18)

n0 = 2
qphi0 = ALL_Qphi[n0]
# img = np.abs(qphi0)
norm = LogNorm(vmin=1, vmax=3e3)
im = ax.imshow(qphi0, cmap='inferno', origin='lower', norm=norm)
plt.colorbar(im, ax=ax, label='Qphi')
ax.set_title(f'Qphi, Cycles: {n0} / {N_CYCLES}')

ax_slider = plt.axes([0.2, 0.05, 0.6, 0.04])
slider = Slider(ax=ax_slider, label='Cycles included',
                valmin=1, valmax=N_CYCLES,
                valinit=n0, valstep=1, valfmt='%d')

def update(val):
    n = int(slider.val)
    qphi = ALL_Qphi[:n]
    im.set_data(np.nanmedian(qphi, axis =0))
    ax.set_title(f'PDI |Qphi| — Cycles: {n} / {N_CYCLES}')
    fig.canvas.draw_idle()

In [57]:
slider.on_changed(update)

plt.show()

In [12]:
test = np.array([np.nanmedian(ALL_Qphi[:i], axis = 0) for i in range(35)])

/Users/bren/miniforge3/envs/pyarm/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


In [13]:
fits.writeto('/Users/bren/Desktop/test.fits', np.nanmedian(ALL_Qphi[:2], axis = 0), overwrite=True)